In [ ]:
!pip install deepxde
!pip install pyDOE
# !pip install efficient-kan
!pip install torchopt

In [ ]:
import time
import torch
from torch import nn
import numpy as np
from torch.func import functional_call, grad, vmap
import torchopt
from collections import OrderedDict
import matplotlib.pyplot as plt
import deepxde as dde
from pyDOE import lhs
from matplotlib.patches import Rectangle
from mpl_toolkits.mplot3d import Axes3D
from torch.optim import LBFGS
from torchsummary import summary

## Dimensiones de varilla en 1D

In [ ]:
T_end = 100.0
T = T_end
L = 1.0
k_diff = 0.003
k = k_diff

training_size = 200
testing_size = 5000

f_x = lambda x: 50*x*(1-x)

torch.manual_seed(123)

def normalize(X):
    X_n = X.clone()
    X_n[:, 0] = X[:, 0] * 2.0 - 1.0          # x  ∈ [0,1]  → [-1,1]
    X_n[:, 1] = X[:, 1] / (T_end / 2.0) - 1.0  # t  ∈ [0,100] → [-1,1]
    return X_n

def LHS_sampling(n, dim, lower, upper):
    return lower + (upper - lower) * lhs(dim, samples=n)


In [ ]:
x = np.linspace(0, 1, 500)

# Función de temperatura inicial
f_x = lambda x: 50 * x * (1 - x)
temperatura = f_x(x)

# Crear la figura con fondo negro
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("black")  # Fondo negro para toda la figura
ax.set_facecolor("black")  # Fondo negro para el área del gráfico

# Graficar la función de temperatura inicial
ax.plot(x, temperatura, color="#1f77b4", lw=3, label="Distribución de temperatura inicial")

# Añadir la varilla (rectángulo gris)
altura_varilla = -5  # Colocar la varilla por debajo del gráfico
ancho_varilla = 1.0
varilla = Rectangle((0, altura_varilla), ancho_varilla, 0.5, color="#5a5a5a", alpha=0.9)
ax.add_patch(varilla)

# Añadir texto para indicar que es la varilla
ax.text(0.5, altura_varilla + 0.2, "Varilla 1D", ha="center", va="center", fontsize=14, color="white")

# Estilizar el gráfico
ax.set_xlim(0, 1)
ax.set_ylim(-10, 30)
ax.axhline(0, color="white", lw=1, linestyle="--")  # Línea base para claridad
ax.set_xlabel("Distancia $x$", fontsize=14, color="white")
ax.set_ylabel("Temperatura $T$", fontsize=14, color="white")
ax.set_title("Varilla 1D con Distribución de Temperatura Inicial", fontsize=16, color="white", pad=20)

# Quitar ticks innecesarios y cambiar colores de los ejes
ax.set_xticks([0, 0.5, 1])
ax.set_xticklabels(["0", "0.5", "1"], fontsize=12, color="white")
ax.set_yticks([0, 10, 20, 30])
ax.set_yticklabels(["0", "10", "20", "30"], fontsize=12, color="white")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color("white")
ax.spines['bottom'].set_color("white")

# Añadir una leyenda

ax.legend(loc="upper right", fontsize=12, facecolor="black", edgecolor="white", labelcolor="white")

# Mostrar el gráfico
plt.tight_layout()
plt.show()

## Generar puntos de boundary conditions

In [ ]:
def generate_boundary_conditions(N):
    x = torch.linspace(0, L, N).view(-1, 1)
    t = torch.linspace(0, T, N).view(-1, 1)
    x, t = torch.meshgrid(x.squeeze(1), t.squeeze(1))

    # initial condition

    # Lo que se hace es agarrar cualquier X y forzar a que el tiempo sea 0
    # en todos los puntos segun la condición inicial u(x, 0) = f(x)
    x_left = torch.hstack((x[:, 0][:, None], t[:, 0][:, None]))
    y_left = f_x(x_left[:, 0]).unsqueeze(1)
    print(y_left.shape)


    # boundary condition

    # Lo que se hace es agarrar cualquier T y forzar a que la x sea 0
    x_bottom = torch.hstack((x[0, :][:, None], t[0, :][:, None]))
    y_bottom = torch.zeros(x_bottom.shape[0], 1)


    # Agarrar cualquier T y forzar a que x sea L
    x_top = torch.hstack((x[-1, :][:, None], t[-1, :][:, None]))
    y_top = torch.zeros(x_top.shape[0], 1)

    X_train = torch.vstack([x_left, x_bottom, x_top])
    y_train = torch.vstack([y_left, y_bottom, y_top])



    return X_train, y_train

X_bc, y_bc = generate_boundary_conditions(training_size)

## Feedforward NN

In [ ]:
class FCN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        activation = nn.Tanh
        layers = OrderedDict([
            ('input', nn.Linear(input_size, hidden_size)),
            ('act0',  activation()),
        ])
        for i in range(num_layers):
            layers[f'h{i}']    = nn.Linear(hidden_size, hidden_size)
            layers[f'act{i+1}'] = activation()
        layers['output'] = nn.Linear(hidden_size, output_size)
        self.net = nn.Sequential(layers)

    def forward(self, x):
        return self.net(x)

## KANlayer
Combina la transformación lineal con la suma de las funciones Splines para crear la activación no lineal.


In [ ]:
class KANLayer(nn.Module):
    """
    Un layer KAN: reemplaza la multiplicación por peso escalar
    con una función spline aprendible por cada par (entrada_i, salida_j).
    """
    def __init__(self, in_features, out_features, grid_size=5, spline_order=3):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.grid_size    = grid_size
        self.k            = spline_order

        # Grid uniforme en [-1, 1] con padding de orden k en cada extremo
        h    = 2.0 / grid_size
        grid = torch.arange(-spline_order, grid_size + spline_order + 1) * h - 1.0
        self.register_buffer('grid', grid)   # no es parámetro, no se optimiza directamente

        # Coeficientes del spline: (out, in, grid_size + k)
        self.spline_weight = nn.Parameter(
            torch.randn(out_features, in_features, grid_size + spline_order) * 0.1
        )
        # Peso de la parte base (SiLU residual, como en el paper original)
        self.base_weight   = nn.Parameter(
            torch.randn(out_features, in_features) * 0.1
        )
        # Escala por neurona
        self.spline_scaler = nn.Parameter(
            torch.ones(out_features, in_features)
        )

    def b_splines(self, x):
        """
        Evalúa las bases B-spline de orden k en los puntos x usando
        la recursión de Cox-de Boor.

        x    : (batch, in_features)
        return: (batch, in_features, grid_size + k)
        """
        x    = x.unsqueeze(-1)          # (batch, in, 1)
        grid = self.grid                # (grid_size + 2k + 1,)

        # Orden 0: función indicadora por intervalo
        bases = ((x >= grid[:-1]) & (x < grid[1:])).float()

        # Recursión hasta orden k
        for order in range(1, self.k + 1):
            left_num  = x - grid[:-(order + 1)]
            left_den  = grid[order:-1] - grid[:-(order + 1)]
            left      = (left_num / (left_den + 1e-8)) * bases[..., :-1]

            right_num = grid[order + 1:] - x
            right_den = grid[order + 1:] - grid[1:-order]
            right     = (right_num / (right_den + 1e-8)) * bases[..., 1:]

            bases = left + right

        return bases  # (batch, in, grid_size + k)

    def forward(self, x):
        """
        x     : (batch, in_features)
        return: (batch, out_features)
        """
        # Parte base: SiLU(x) @ W_base^T
        base_out   = torch.nn.functional.silu(x) @ self.base_weight.T

        # Parte spline: contracción sobre (in_features, nodos)
        splines    = self.b_splines(x)  # (batch, in, grid+k)
        scaled_w   = self.spline_weight * self.spline_scaler.unsqueeze(-1)
        spline_out = torch.einsum('bik,oik->bo', splines, scaled_w)

        return base_out + spline_out


class KAN(nn.Module):
    def __init__(self, layers_hidden, grid_size=5, spline_order=3):
        """
        layers_hidden: e.g. [2, 10, 10, 1]
        """
        super().__init__()
        self.layers = nn.ModuleList([
            KANLayer(in_f, out_f, grid_size, spline_order)
            for in_f, out_f in zip(layers_hidden[:-1], layers_hidden[1:])
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

## Pérdida física
Se integran las ecuaciones diferenciales. En este caso, la ecuación de calor en 1D.



In [ ]:
loss_fn = nn.MSELoss(reduction='mean')

def physics_loss(X, model, normalize_fn=None):
    x_raw, t_raw = X[:, 0:1], X[:, 1:2]
    inp = torch.hstack((x_raw, t_raw))
    if normalize_fn is not None:
        inp = normalize_fn(inp)
    u     = model(inp)
    u_t   = torch.autograd.grad(u, t_raw, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x   = torch.autograd.grad(u, x_raw, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx  = torch.autograd.grad(u_x, x_raw, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    res   = u_t - k_diff * u_xx
    return torch.mean(res ** 2)

## Pérdida de BCs
Aquí se usa un enfoque tradicional de una red neuronal, donde se ingresan las coordenadas x y el tiempo, el ground truth está establecido en el dataset y se compara el output de la red con el ground truth.

In [ ]:
def boundary_loss(model, X_bc, y_bc, normalize_fn=None):
    inp = normalize_fn(X_bc) if normalize_fn is not None else X_bc
    u   = model(inp)
    return loss_fn(u, y_bc)

## Pérdida de PINN
Se integra la pérdida física y la pérdida de BCs para asegurar que la red neuronal aprenda tanto las ecuaciones diferenciales como las condiciones de frontera.

In [ ]:
def total_loss(X, model, w_ph, w_bc, normalize_fn=None):
    return (w_ph * physics_loss(X, model, normalize_fn)
            + w_bc * boundary_loss(model, X_bc, y_bc, normalize_fn))


## Puntos de colocación
Se generan puntos de colocación para evaluar la ecuación de calor en 1D y minimizar la phyisics loss a 0.

In [ ]:
def generate_collocation_points(N=2000):
    samples = lhs(2, samples=N)
    x_col   = torch.tensor(samples[:, 0] * L,    dtype=torch.float32)
    t_col   = torch.tensor(samples[:, 1] * T_end, dtype=torch.float32)
    return torch.stack([x_col, t_col], dim=1).requires_grad_(True)

def generate_collocation_points_stratified(N_early=1000, N_late=500):
    """67% de puntos en t∈[0,10] donde la dinámica es rápida."""
    s_e = lhs(2, samples=N_early)
    x_e = torch.tensor(s_e[:, 0] * L,    dtype=torch.float32)
    t_e = torch.tensor(s_e[:, 1] * 10.0, dtype=torch.float32)  # [0, 10]

    s_l = lhs(2, samples=N_late)
    x_l = torch.tensor(s_l[:, 0] * L,                   dtype=torch.float32)
    t_l = torch.tensor(10.0 + s_l[:, 1] * 90.0,          dtype=torch.float32)  # [10, 100]

    x_all = torch.cat([x_e, x_l])
    t_all = torch.cat([t_e, t_l])
    return torch.stack([x_all, t_all], dim=1).requires_grad_(True)

## Entrenamiento

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# @title
import tqdm
from tqdm.notebook import tqdm
import csv, os, time

def train_model(model, X_train, num_epochs=5000, lr=0.01,
                normalize_fn=None, label='', checkpoint_every=1000,
                w_ph=1.0, w_bc=1.0):
    """
    Entrena el modelo con progress bar y guarda:
      - CSV de losses:     f'{label}_losses.csv'
      - Checkpoint .pth:   f'{label}_checkpoint_{epoch}.pth'  (cada checkpoint_every epochs)
      - Checkpoint final:  f'{label}_final.pth'
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS,
        eta_min=1e-5
    )
    losses    = []
    t0        = time.time()

    # Preparar CSV
    csv_path = f'{label}_losses.csv'
    csv_file = open(csv_path, 'w', newline='')
    writer   = csv.writer(csv_file)
    writer.writerow(['epoch', 'loss'])

    # Progress bar
    pbar = tqdm(range(num_epochs), desc=f'{label}', unit='ep',
                bar_format='{l_bar}{bar:30}{r_bar}')

    for epoch in pbar:
        if epoch % 1000 == 0:
            X_new      = generate_collocation_points_stratified()
            X_train    = X_new          # actualiza la base
            X_train_ep = X_new
        elif epoch % 200 == 0:
            # Cada 200 epochs, mezcla 20% de puntos nuevos con 80% existentes
            X_fresh    = generate_collocation_points_stratified(
                            N_early=200, N_late=100)   # 20% del total
            X_train_ep = torch.cat([
                X_train[:1200].detach().requires_grad_(True),
                X_fresh
            ], dim=0)
        else:
            X_train_ep = X_train


        optimizer.zero_grad()
        loss = total_loss(X_train_ep, model, w_ph=w_ph, w_bc=w_bc,
                          normalize_fn=normalize_fn)
        loss.backward()
        optimizer.step()
        loss_val = loss.item()
        scheduler.step()

        losses.append(loss_val)
        writer.writerow([epoch, loss_val])

        # Actualizar barra con loss actual
        pbar.set_postfix({'loss': f'{loss_val:.5f}'})

        # Checkpoint periódico
        if checkpoint_every and (epoch + 1) % checkpoint_every == 0:
            torch.save(model.state_dict(),
                       f'{label}_checkpoint_{epoch+1}.pth')

    csv_file.close()
    elapsed = time.time() - t0

    # Checkpoint final
    torch.save(model.state_dict(), f'{label}_final.pth')
    print(f'[{label}] {elapsed:.1f}s | Loss final: {losses[-1]:.6f} '
          f'| Guardado: {csv_path}, {label}_final.pth')

    return losses, elapsed



In [ ]:
import pandas as pd

#  PIKAN: cargar desde checkpoint, no reentrenar
model_kan = KAN([2, 10, 10, 1], grid_size=5, spline_order=3)
model_kan.load_state_dict(torch.load('PIKAN_final.pth'))
model_kan.eval()
df_kan     = pd.read_csv('PIKAN_losses.csv')
losses_kan = df_kan['loss'].tolist()
time_kan   = 2245.7
print(f'PIKAN cargado | Parámetros: {sum(p.numel() for p in model_kan.parameters()):,}')

model_pinn = FCN(2, 20, 2, 1)
print(f'PINN  | Parámetros: {sum(p.numel() for p in model_pinn.parameters()):,}')

NUM_EPOCHS = 5000
X_train    = generate_collocation_points_stratified()

losses_pinn, time_pinn = train_model(
    model_pinn, X_train,
    num_epochs=NUM_EPOCHS,
    lr=0.001,
    w_ph=1.0,
    w_bc=1.0,
    normalize_fn=None,
    label='PINN',
    checkpoint_every=1000
)

In [ ]:

nx_fd = 50
nt_fd = 2000
dt_fd = 0.05
dx_fd = L / (nx_fd - 1)

x_fd  = np.linspace(0, L, nx_fd)
u_fd  = np.zeros((nt_fd, nx_fd))
u_fd[0, :] = 50 * x_fd * (1 - x_fd)

for n in range(1, nt_fd):
    for i in range(1, nx_fd - 1):
        u_fd[n, i] = (u_fd[n-1, i]
                      + k_diff * dt_fd / dx_fd**2
                      * (u_fd[n-1, i+1] - 2*u_fd[n-1, i] + u_fd[n-1, i-1]))

t_fd_axis = np.linspace(0, nt_fd * dt_fd, nt_fd)  # 0 … 100

# Evaluar PINN y PIKAN en la misma malla que FD
T_fd_grid, X_fd_grid = np.meshgrid(t_fd_axis, x_fd)   # (nx, nt)
XT_fd = torch.tensor(
    np.column_stack([X_fd_grid.ravel(), T_fd_grid.ravel()]),
    dtype=torch.float32
)

with torch.no_grad():
    u_pinn_fd = model_pinn(XT_fd).numpy().reshape(nx_fd, nt_fd)
    u_kan_fd  = model_kan(normalize(XT_fd)).numpy().reshape(nx_fd, nt_fd)

u_fd_T = u_fd.T

## Métricas

In [ ]:
def metrics(u_pred_np, u_true_np):
    """u_pred, u_true: numpy arrays de la misma forma"""
    diff      = u_pred_np - u_true_np
    l2_abs    = np.linalg.norm(diff) / diff.size**0.5          # L2 por elemento
    l2_rel    = (np.linalg.norm(diff)
                 / (np.linalg.norm(u_true_np) + 1e-8))        # L2 relativa global
    max_err   = np.abs(diff).max()
    return l2_abs, l2_rel, max_err

l2_pinn, l2r_pinn, max_pinn = metrics(u_pinn_fd, u_fd_T)
l2_kan,  l2r_kan,  max_kan  = metrics(u_kan_fd,  u_fd_T)

plt.style.use('default')
CMAP = 'viridis'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Pérdida durante Entrenamiento', fontsize=15, fontweight='bold')

for ax, losses, label, color in zip(
        axes,
        [losses_pinn, losses_kan],
        ['PINN (FCN)', 'PIKAN (KAN)'],
        ['steelblue', 'darkorange']):
    ax.semilogy(losses, color=color, lw=1.5)
    ax.set_xlabel('Época', fontsize=12)
    ax.set_ylabel('Pérdida (log)', fontsize=12)
    ax.set_title(label, fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, NUM_EPOCHS)

plt.tight_layout()
plt.savefig('loss_curves.pdf', bbox_inches='tight')
plt.show()


In [ ]:
N_eval = 200
x_ev = np.linspace(0, L, N_eval)
t_ev = np.linspace(0, T_end, N_eval)
T_ev, X_ev = np.meshgrid(t_ev, x_ev)   # (N_eval, N_eval)

XT_ev = torch.tensor(
    np.column_stack([X_ev.ravel(), T_ev.ravel()]), dtype=torch.float32
)

with torch.no_grad():
    u_pinn_surf = model_pinn(XT_ev).numpy().reshape(N_eval, N_eval)
    u_kan_surf  = model_kan(normalize(XT_ev)).numpy().reshape(N_eval, N_eval)

vmin = min(u_pinn_surf.min(), u_kan_surf.min(), u_fd_T.min())
vmax = max(u_pinn_surf.max(), u_kan_surf.max(), u_fd_T.max())

fig = plt.figure(figsize=(18, 5))
fig.suptitle('Evolución de Temperatura — Comparación de Métodos', fontsize=15, fontweight='bold')

titles   = ['Diferencias Finitas (Referencia)', 'PINN', 'PIKAN']
surfaces = [
    (np.meshgrid(t_fd_axis, x_fd)[0], np.meshgrid(t_fd_axis, x_fd)[1], u_fd_T),
    (T_ev, X_ev, u_pinn_surf),
    (T_ev, X_ev, u_kan_surf),
]

for idx, (title, (T_s, X_s, U_s)) in enumerate(zip(titles, surfaces)):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    surf = ax.plot_surface(T_s, X_s, U_s, cmap=CMAP, vmin=vmin, vmax=vmax)
    ax.view_init(elev=25, azim=-50)
    ax.set_xlabel('Tiempo', fontsize=10)
    ax.set_ylabel('Espacio', fontsize=10)
    ax.set_zlabel('T', fontsize=10)
    ax.set_title(title, fontsize=12)
    ax.set_zlim(vmin - 0.5, vmax + 0.5)
    fig.colorbar(surf, ax=ax, shrink=0.4, aspect=8)

plt.tight_layout()
plt.savefig('temperature_evolution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
from scipy.interpolate import RegularGridInterpolator

interp_fd = RegularGridInterpolator(
    (x_fd, t_fd_axis), u_fd_T, method='linear', bounds_error=False, fill_value=None
)
pts = np.column_stack([X_ev.ravel(), T_ev.ravel()])
u_fd_interp = interp_fd(pts).reshape(N_eval, N_eval)

err_pinn = np.abs(u_pinn_surf - u_fd_interp)
err_kan  = np.abs(u_kan_surf  - u_fd_interp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Error Absoluto respecto a Diferencias Finitas', fontsize=15, fontweight='bold')

emax = max(err_pinn.max(), err_kan.max())
for ax, err, label, color in zip(
        axes,
        [err_pinn, err_kan],
        ['PINN', 'PIKAN'],
        ['Blues', 'Oranges']):
    im = ax.pcolormesh(T_ev, X_ev, err, cmap=color, vmin=0, vmax=emax, shading='auto')
    ax.set_xlabel('Tiempo', fontsize=12)
    ax.set_ylabel('Espacio', fontsize=12)
    ax.set_title(f'|{label} − FD|', fontsize=13)
    plt.colorbar(im, ax=ax, label='Error absoluto')

plt.tight_layout()
plt.savefig('error_heatmap.pdf', bbox_inches='tight')
plt.show()

## Comparación de resultados con método de diferencias finitas

In [ ]:
print('\n' + '='*65)
print(f'{"Métrica":<30} {"PINN":>15} {"PIKAN":>15}')
print('-'*65)
print(f'{"L2 absoluta media":<30} {l2_pinn:>15.6f} {l2_kan:>15.6f}')
print(f'{"L2 relativa global":<30} {l2r_pinn:>15.6f} {l2r_kan:>15.6f}')
print(f'{"Error máximo":<30} {max_pinn:>15.6f} {max_kan:>15.6f}')
print(f'{"Tiempo entrenamiento (s)":<30} {time_pinn:>15.1f} {time_kan:>15.1f}')
print(f'{"Pérdida final":<30} {losses_pinn[-1]:>15.6f} {losses_kan[-1]:>15.6f}')
print(f'{"Parámetros":<30} {sum(p.numel() for p in model_pinn.parameters()):>15,} '
      f'{sum(p.numel() for p in model_kan.parameters()):>15,}')
print('='*65)

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

col_labels = ['Métrica', 'PINN', 'PIKAN']
row_data = [
    ['L2 absoluta media',     f'{l2_pinn:.6f}',  f'{l2_kan:.6f}'],
    ['L2 relativa global',    f'{l2r_pinn:.6f}', f'{l2r_kan:.6f}'],
    ['Error máximo',          f'{max_pinn:.6f}', f'{max_kan:.6f}'],
    ['Tiempo entren. (s)',     f'{time_pinn:.1f}', f'{time_kan:.1f}'],
    ['Pérdida final',         f'{losses_pinn[-1]:.6f}', f'{losses_kan[-1]:.6f}'],
    ['Parámetros',
     f'{sum(p.numel() for p in model_pinn.parameters()):,}',
     f'{sum(p.numel() for p in model_kan.parameters()):,}'],
]

table = ax.table(
    cellText=row_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.8)

# Color header
for j in range(3):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Color alternado filas
for i in range(1, len(row_data) + 1):
    bg = '#f0f0f0' if i % 2 == 0 else 'white'
    for j in range(3):
        table[i, j].set_facecolor(bg)

plt.title('Resumen de Comparación: PINN vs PIKAN', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('comparison_table.pdf', bbox_inches='tight')
plt.show()


##  Perfiles temporales en x=0.5
La más informativa. Muestra la evolución de temperatura en el punto central de la varilla para los tres métodos en la misma figura 2D:



In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Evaluar en x=0.5 para todo t
t_line  = np.linspace(0, T_end, 500)
x_fixed = np.full_like(t_line, 0.5)
XT_line = torch.tensor(np.column_stack([x_fixed, t_line]), dtype=torch.float32)

with torch.no_grad():
    u_pinn_line = model_pinn(XT_line).numpy().flatten()
    u_kan_line  = model_kan(normalize(XT_line)).numpy().flatten()

# FD en x=0.5 — índice 25 de 50
from scipy.interpolate import interp1d
fd_at_half = interp1d(t_fd_axis, u_fd_T[25, :], kind='linear')
u_fd_line  = fd_at_half(t_line)

ax.plot(t_line, u_fd_line,   'k-',  lw=2,   label='FD (referencia)')
ax.plot(t_line, u_pinn_line, 'b--', lw=1.5, label='PINN')
ax.plot(t_line, u_kan_line,  'r-',  lw=1.5, label='PIKAN')
ax.set_xlabel('Tiempo', fontsize=13)
ax.set_ylabel('Temperatura en x=0.5', fontsize=13)
ax.set_title('Perfil temporal en el punto central (x = 0.5)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('perfil_temporal.pdf', bbox_inches='tight')
plt.show()

## Perfiles espaciales en t=0, t=5, t=20

Snapshots de la distribución espacial en momentos clave ,revela si el error está en la condición inicial o en la difusión:


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Perfiles espaciales en distintos tiempos', fontsize=14, fontweight='bold')

x_line  = np.linspace(0, L, 200)
t_snaps = [0.1, 5.0, 20.0]   # t=0 exacto da división por cero en FD, usar 0.1

for ax, t_snap in zip(axes, t_snaps):
    XT_snap = torch.tensor(
        np.column_stack([x_line, np.full_like(x_line, t_snap)]),
        dtype=torch.float32
    )
    with torch.no_grad():
        u_p = model_pinn(XT_snap).numpy().flatten()
        u_k = model_kan(normalize(XT_snap)).numpy().flatten()

    # FD interpolado
    t_idx = np.argmin(np.abs(t_fd_axis - t_snap))
    u_f   = interp1d(x_fd, u_fd_T[:, t_idx], kind='linear')(x_line)

    ax.plot(x_line, u_f, 'k-',  lw=2,   label='FD')
    ax.plot(x_line, u_p, 'b--', lw=1.5, label='PINN')
    ax.plot(x_line, u_k, 'r-',  lw=1.5, label='PIKAN')
    ax.set_title(f't = {t_snap}', fontsize=12)
    ax.set_xlabel('Espacio x')
    ax.set_ylabel('Temperatura')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('perfiles_espaciales.pdf', bbox_inches='tight')
plt.show()

## Error relativo por tiempo (integrado sobre x)
Muestra en qué región temporal cada modelo falla más
,diagnóstico directo del spectral bias


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

t_eval_diag = np.linspace(0, T_end, 200)
err_pinn_t  = []
err_kan_t   = []

for t_val in t_eval_diag:
    XT_t = torch.tensor(
        np.column_stack([x_line, np.full_like(x_line, t_val)]),
        dtype=torch.float32
    )
    with torch.no_grad():
        u_p = model_pinn(XT_t).numpy().flatten()
        u_k = model_kan(normalize(XT_t)).numpy().flatten()

    t_idx = np.argmin(np.abs(t_fd_axis - t_val))
    u_f   = interp1d(x_fd, u_fd_T[:, t_idx], kind='linear')(x_line)

    # Error L2 relativo en este t
    denom = np.linalg.norm(u_f) + 1e-8
    err_pinn_t.append(np.linalg.norm(u_p - u_f) / denom)
    err_kan_t.append( np.linalg.norm(u_k - u_f) / denom)

ax.semilogy(t_eval_diag, err_pinn_t, 'b-',  lw=1.5, label='PINN')
ax.semilogy(t_eval_diag, err_kan_t,  'r-',  lw=1.5, label='PIKAN')
ax.axvline(10, color='gray', linestyle='--', alpha=0.5, label='t=10 (límite estratificado)')
ax.set_xlabel('Tiempo', fontsize=13)
ax.set_ylabel('Error L2 relativo (log)', fontsize=13)
ax.set_title('Error relativo por franja temporal', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('error_por_tiempo.pdf', bbox_inches='tight')
plt.show()